# GenAI Platform — Demo

Este notebook demonstra a plataforma camada por camada.

```
ADKRuntime  ←  mesmo Tool, zero mudança de business logic
    ↓
CandidateSearchTool   execute() + schema()
    ↓
CandidateRetriever
    ↓
EmbeddingProvider     GeminiEmbeddingProvider
VectorStore           FAISSVectorStore
```

**Regra de validação:** trocar qualquer provider muda só o wiring. Nunca a business logic.

## 0. Setup

In [4]:
!pip install -q -U google-genai google-adk pydantic python-dotenv faiss-cpu requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
import sys

# Colab: clona o repositório e aponta o sys.path
# Local: apenas adiciona a raiz do repo ao sys.path
try:
    from google.colab import userdata  # só existe no Colab
    if not os.path.exists('/content/agent_workshop'):
        !git clone https://github.com/Willblemos2/agent_workshop.git /content/agent_workshop
    repo_root = '/content/agent_workshop'
except ModuleNotFoundError:
    # Sobe dois níveis a partir de notebooks/workshops/
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f'repo_root: {repo_root}')

repo_root: c:\Users\willi\Documents\Projects\agent_workshop


In [6]:
# Colab: lê do Secrets (menu lateral → chave 🔑 → GOOGLE_API_KEY)
# Local: lê do .env na raiz do repositório
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key carregada do Colab Secrets.')
except ModuleNotFoundError:
    from src.shared.config import load_env
    load_env()
    print('API key carregada do .env local.')

API key carregada do .env local.


## 1. Dataset

Carregamos o dataset já processado — embeddings Gemini (3072 dim) pré-computados.
Não há chamada de API nesta etapa.

In [7]:
from src.shared.dataset import load_processed_candidates

candidates_df = load_processed_candidates()

print(f'Candidatos carregados: {len(candidates_df)}')
candidates_df[['candidate_id', 'first_name', 'last_name', 'content']].head(3)

Candidatos carregados: 40


,candidate_id,first_name,last_name,content
0,c34b7d23-63db-4593-9820-c0a432181152,Taylor,Miller,MORTGAGE BANKING FORECLOSURE SPECIALIST ...
1,4bccf414-c958-49a0-9818-d5c3b38494c9,Drew,Clark,SOUS CHEF Work Experience Sous Chef...
2,2987c661-b14c-4973-b901-5bf5aa3b6d67,Jamie,Thomas,MEDIA ACTIVITIES SPECIALIST Summary ...


## 2. Foundation — Vector Store

Criamos o índice FAISS e inserimos os candidatos usando os embeddings pré-computados.

In [10]:
from src.foundation.vector_store import FAISSVectorStore, build_records
from src.foundation.vector_store.schemas import CANDIDATE_SCHEMA

store = FAISSVectorStore()
store.create_index(index_name='candidates_index', schema=CANDIDATE_SCHEMA)

records = build_records(
    rows=candidates_df.to_dict(orient='records'),
    id_field='candidate_id',
)
store.insert(index_name='candidates_index', records=records)

print(f'{len(records)} candidatos indexados no FAISS.')

40 candidatos indexados no FAISS.


## 3. Foundation — Embedding Provider

O provider é injetado — o retriever não sabe qual está usando.

In [11]:
from src.foundation.embeddings import GeminiEmbeddingProvider

embedding_provider = GeminiEmbeddingProvider()

sample = embedding_provider.embed('Senior Python engineer')
print(f'Embedding gerado: {len(sample)} dimensões')

Embedding gerado: 3072 dimensões


## 4. Application — Retriever

`CandidateRetriever` não conhece Gemini, não conhece FAISS.
Recebe os dois por injeção — troca de provider não muda esta camada.

In [12]:
from src.applications.retrieval import CandidateRetriever

retriever = CandidateRetriever(
    vector_store=store,
    embedding_provider=embedding_provider,
)

results = retriever.search('Python developer with machine learning experience', top_k=3)

for r in results:
    print(f'[{r.score:.4f}] {r.first_name} {r.last_name} — {r.content_preview[:80]}...')

[0.5990] Taylor Anderson — WORKING         RF SYSTEMS ENGINEER             Qualifications        Microsoft ...
[0.5717] Cameron Miller — REGISTERED CLIENT SERVICE ASSOCIATE       Summary    To obtain a position where ...
[0.5697] Jordan Clark — AVIATION ELECTRONICS TECHNICIAN       Summary    Seeking a challenging position ...


## 5. Tool Layer

O `Tool` expõe dois contratos:
- `execute()` — roda a capability
- `schema()` — descreve a tool para qualquer runtime

O runtime lê `schema()` para registrar a tool no framework. Nunca toca a business logic.

In [13]:
from src.tools import CandidateSearchTool

search_tool = CandidateSearchTool(retriever=retriever)

# schema() — o que qualquer runtime vai ler
schema = search_tool.schema()
print(f'Tool: {schema.name}')
print(f'Descrição: {schema.description}')
print(f'Parâmetros:')
for p in schema.parameters:
    req = 'obrigatório' if p.required else 'opcional'
    print(f'  - {p.name} ({p.type}, {req}): {p.description}')

Tool: candidate_search
Descrição: Search for candidates matching a natural language query. Returns a ranked list of candidates with profile information.
Parâmetros:
  - query (string, obrigatório): Natural language description of the ideal candidate.
  - top_k (integer, opcional): Maximum number of candidates to return. Defaults to 10.


In [14]:
# execute() — chamada direta, sem agent
result = search_tool.execute(query='data scientist with NLP skills', top_k=3)

if result.error:
    print(f'Erro: {result.error}')
else:
    for c in result.content:
        print(f"[{c['score']:.4f}] {c['first_name']} {c['last_name']}")

[0.5748] Cameron Anderson
[0.5731] Taylor Anderson
[0.5724] Cameron Miller


## 6. Runtime — ADK Agent

O mesmo `CandidateSearchTool` agora roda dentro do ADK.
O `ADKRuntime` lê `tool.schema()` e converte para o formato do ADK automaticamente.

Zero mudança na tool, zero mudança no retriever.

In [15]:
from src.runtime import ADKRuntime

runtime = ADKRuntime(
    model='gemini-2.5-flash',
    instruction='You are a recruitment assistant. Use the candidate_search tool to find relevant candidates and summarize your findings clearly.',
)

response = await runtime.run(
    input='Find me 3 software engineers with cloud infrastructure experience.',
    tools=[search_tool],
)

print(response)

I found 3 candidates:

*   **Taylor Anderson**: RF Systems Engineer with experience in Microsoft Office, RF/Microwave software (Ansoft Designer, AWR Microwave Office), and RF test.
*   **Cameron Miller**: Information Technology Technician I / Systems Administrator with strong troubleshooting skills for networking, end-user problems, and network security.
*   **Cameron Johnson**: Sr. Estimator-BAS Construction Sales with 9+ years of experience in energy solution consulting, performance contracting, HVAC estimating & designing.

Please note that none of the top 3 candidates explicitly mention "cloud infrastructure" in their `content_preview`. The search results might be broad. If you'd like me to refine the search or look for other candidates, please let me know!I found the following 3 candidates:

1.  **Taylor Anderson**: An RF Systems Engineer with experience in RF/Microwave software and testing.
2.  **Cameron Miller**: An Information Technology Technician I / Systems Administrator wit

In [16]:
# Stream — mesma tool, mesma lógica, output em tempo real
async for chunk in runtime.stream(
    input='Who are the best candidates for a senior data engineering role?',
    tools=[search_tool],
):
    print(chunk, end='', flush=True)

Based on my search, the top candidates do not appear to be directly aligned with a Senior Data Engineering role based on their content previews. Here are the top 3 candidates:

*   **Taylor Anderson:** RF Systems Engineer
*   **Morgan Thomas:** Equipment Engineering Technician
*   **Cameron Johnson:** Sr. Estimator-BAS Construction Sales

The current search results do not show strong matches for the requested role. Would you like me to refine the search with more specific keywords or look for a different role?

## 7. Harness — Troca de Provider

Validação arquitetural: trocar FAISS por Milvus não muda nenhuma linha de business logic.

> **Compatibilidade:** `milvus-lite` (conexão local via arquivo) requer **Linux ou macOS**.
> No Windows, esta seção exige um servidor Milvus rodando via Docker.
> No Colab (Linux), funciona diretamente.

In [ ]:
!pip install -q "pymilvus>=2.4.0,<2.5"

In [ ]:
try:
    # Import direto do módulo — bypassa o cache do __init__.py
    # (o módulo pai foi cacheado antes do pymilvus estar instalado)
    from src.foundation.vector_store.milvus_store import MilvusVectorStore

    milvus_store = MilvusVectorStore()
    milvus_store.create_index(index_name='candidates_index', schema=CANDIDATE_SCHEMA)
    milvus_store.insert(index_name='candidates_index', records=records)

    milvus_retriever = CandidateRetriever(
        vector_store=milvus_store,
        embedding_provider=embedding_provider,
    )

    milvus_results = milvus_retriever.search('Python developer with machine learning experience', top_k=3)

    print('Resultados via Milvus:')
    for r in milvus_results:
        print(f'[{r.score:.4f}] {r.first_name} {r.last_name}')

    print('\n✅ Harness validado: FAISS → Milvus sem mudança de business logic.')

except Exception as e:
    print(f'⚠️  Milvus indisponível neste ambiente ({type(e).__name__}: {e})')
    print()
    print('Causas comuns:')
    print('  - Python 3.12+: instale setuptools (pip install setuptools)')
    print('  - Conflito de grpcio entre pymilvus e google-adk')
    print()
    print('A validação arquitetural permanece válida — apenas o wiring muda.')
    print('Rode em ambiente limpo (ex: novo venv ou Colab) para confirmar.')

## 8. Chat Loop

Conversa em tempo real com o agent. A resposta é streamada chunk a chunk.
Digite `sair` para encerrar.

In [ ]:
print("Recruitment Agent — Chat")
print("=" * 40)
print("Comandos: 'sair' para encerrar | 'reset' para limpar o histórico\n")

while True:
    user_input = input("Você: ").strip()

    if not user_input:
        continue

    if user_input.lower() in ("sair", "exit", "quit"):
        print("Encerrando.")
        break

    if user_input.lower() == "reset":
        runtime.reset()
        print("— histórico limpo —\n")
        continue

    # Ecoa a mensagem no output da célula (input() aparece no topo no VS Code)
    print(f"Você: {user_input}")
    print("\nAgent: ", end="", flush=True)
    async for chunk in runtime.stream(user_input, tools=[search_tool]):
        print(chunk, end="", flush=True)
    print("\n")